In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 9: ML Monitoring & Distribution Shift Detection

### Business Context
A deployed CTR model was trained in January. It's now July. Click rates are dropping. Is this a business problem (users less engaged) or a model problem (distribution shift)? You need automated monitoring to detect and triage.

### Types of Drift

| Type | What Changes | Example | Detection |
|------|-------------|---------|----------|
| Data drift (covariate shift) | P(X) changes, P(Y\|X) same | User demographics shift | PSI on features |
| Concept drift | P(Y\|X) changes, P(X) same | User preferences change | Monitor predictions vs labels |
| Label drift | P(Y) changes | Overall click rate seasonality | Track label distribution |
| Feature drift | One feature goes stale | Partner API stops sending data | Null rates, value ranges |

### Population Stability Index (PSI)
Measures distribution shift between training and serving feature distributions:
```
PSI = Σ (actual% - expected%) * ln(actual% / expected%)

PSI < 0.1:  no significant change
0.1-0.2:    moderate change — investigate
PSI > 0.2:  significant shift — consider retraining
```

In [ ]:
def calculate_psi(expected, actual, n_bins=10, epsilon=1e-4):
    """
    Population Stability Index.
    expected: training distribution
    actual:   serving distribution
    """
    # Use training distribution to define bin edges
    breakpoints = np.quantile(expected, np.linspace(0, 1, n_bins + 1))
    breakpoints[0]  -= 1e-8   # ensure min is included
    breakpoints[-1] += 1e-8   # ensure max is included

    expected_pcts = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_pcts   = np.histogram(actual,   bins=breakpoints)[0] / len(actual)

    # Clip to avoid log(0)
    expected_pcts = np.clip(expected_pcts, epsilon, 1)
    actual_pcts   = np.clip(actual_pcts,   epsilon, 1)

    psi = np.sum((actual_pcts - expected_pcts) * np.log(actual_pcts / expected_pcts))
    return psi

# Simulate 3 drift scenarios
np.random.seed(42)
n = 5000

training     = np.random.normal(0, 1, n)          # baseline
no_drift     = np.random.normal(0, 1, n)          # no change
mild_drift   = np.random.normal(0.3, 1.1, n)      # small shift
severe_drift = np.random.normal(1.5, 1.5, n)      # large shift

print('PSI Scores (threshold: 0.1=investigate, 0.2=retrain):')
print(f'  No drift:     {calculate_psi(training, no_drift):.4f}')
print(f'  Mild drift:   {calculate_psi(training, mild_drift):.4f}')
print(f'  Severe drift: {calculate_psi(training, severe_drift):.4f}')

# --- KL Divergence for prediction distribution monitoring ---
def kl_divergence(p, q, n_bins=20, epsilon=1e-8):
    breakpoints = np.linspace(min(p.min(), q.min()), max(p.max(), q.max()), n_bins + 1)
    p_dist = np.histogram(p, bins=breakpoints)[0] / len(p) + epsilon
    q_dist = np.histogram(q, bins=breakpoints)[0] / len(q) + epsilon
    return np.sum(p_dist * np.log(p_dist / q_dist))

train_preds   = np.random.beta(2, 5, n)     # training score distribution
serving_preds = np.random.beta(2, 5, n)     # no shift
shifted_preds = np.random.beta(4, 3, n)     # shifted (model outputs higher scores)

print(f'\nKL Divergence on predictions:')
print(f'  No shift:  {kl_divergence(train_preds, serving_preds):.4f}')
print(f'  Shifted:   {kl_divergence(train_preds, shifted_preds):.4f}')

# --- Monitoring dashboard summary ---
def drift_report(feature_psi_scores, prediction_kl, label_rate_change_pct):
    print('\n=== Drift Monitoring Report ===')
    drifted_features = [(f, v) for f, v in feature_psi_scores.items() if v > 0.1]
    print(f'Features with drift: {len(drifted_features)}/{len(feature_psi_scores)}')
    for f, v in sorted(drifted_features, key=lambda x: -x[1]):
        severity = 'SEVERE' if v > 0.2 else 'MODERATE'
        print(f'  {f}: PSI={v:.3f} [{severity}]')
    print(f'Prediction distribution KL: {prediction_kl:.4f} {"[ALERT]" if prediction_kl > 0.1 else "[OK]"}')
    print(f'Label rate change: {label_rate_change_pct:+.1f}% {"[ALERT]" if abs(label_rate_change_pct) > 10 else "[OK]"}')

drift_report(
    {'age': 0.05, 'session_depth': 0.23, 'device_type': 0.08, 'days_since_last_visit': 0.31},
    prediction_kl=0.07,
    label_rate_change_pct=-12.5
)

### L6 Interview Q&A

**Q: A model's AUC is stable but business metrics are dropping. What do you investigate?**  
A: AUC measures ranking quality independent of calibration. Possible causes: (1) label drift — overall click rate dropped seasonally, model still ranks well but the threshold is wrong; (2) prediction distribution shifted — model is well-calibrated on training data but miscalibrated on current serving traffic; (3) a high-traffic segment the model was accurate on has drifted. Start by checking PSI on features, then KL on prediction distribution, then segment-level performance.

**Q: Explain the difference between data drift and concept drift.**  
A: Data drift (covariate shift): the input feature distribution P(X) changes but the relationship P(Y|X) remains the same. The model would perform fine if it saw the new distribution. Concept drift: P(Y|X) changes — the relationship between features and labels has fundamentally changed. More serious — retraining on old data won't help; you need new labels. In practice, distinguish them by checking whether features or labels (or both) have shifted.

**Q: What is shadow mode and when do you use it?**  
A: Shadow mode runs a new model in parallel with the production model — the new model scores every request but its outputs are NOT shown to users. You collect shadow predictions and compare them to the production model's predictions and (eventually) to outcomes. Use it to: validate a new model's behavior at full production traffic before cutover, detect if a new model would behave very differently, estimate online metrics without risk.

### Common Pitfalls
- Monitoring only overall metrics — segment-level drift can be invisible in aggregate
- Not monitoring null rates and value ranges per feature — silent data pipeline failures
- Setting PSI thresholds too tight — alert fatigue; too loose — misses real issues